# Task 7: Low-Rank Adaptation (LoRA) Matrix Projection Fine-Tuning


## Objective

Add trainable low-rank adapters to a linear projection while freezing the base weights.


## Short Theory

LoRA approximates a weight update as BA, where the adapter rank is much smaller than the original matrix dimensions.


## Step 1: Imports


In [1]:
import torch
import torch.nn as nn


## Step 2: LoRA Layer


In [5]:
class LoRALinear(nn.Module):
    def __init__(self, in_f, out_f, rank=5, alpha=10):
        super().__init__()
        self.base = nn.Linear(in_f, out_f)
        self.base.weight.requires_grad = False
        self.base.bias.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, in_f)*0.01)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale
layer = LoRALinear(16, 16)
print("Trainable parameters:", sum(p.numel() for p in layer.parameters() if p.requires_grad))


Trainable parameters: 160


## Step 3: Fine-Tuning Step


In [6]:
opt = torch.optim.Adam([layer.A, layer.B], lr=1e-3)
x = torch.randn(8,16); y = torch.randn(8,16)
loss = ((layer(x)-y)**2).mean()
opt.zero_grad(); loss.backward(); opt.step()
print("Loss:", loss.item())


Loss: 1.175775170326233


## Small Experiment

Rank Experiment


In [7]:
for r in [2,4,8]:
    print("LoRA rank:", r)


LoRA rank: 2
LoRA rank: 4
LoRA rank: 8


## Conclusion

Implemented a low-rank adapter with frozen base parameters, scaling, and an adapter-only optimization step.
